In [2]:
!pip install numpy scipy pandas yfinance plotly streamlit -q
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    """
    Black-Scholes option pricing formula.

    Parameters:
        S     : Current stock price
        K     : Strike price
        T     : Time to expiration (in years)
        r     : Risk-free interest rate (decimal, e.g. 0.05 = 5%)
        sigma : Volatility (decimal, e.g. 0.2 = 20%)
        option_type : 'call' or 'put'

    Returns:
        price : Option price
    """
    if T <= 0:
        if option_type == 'call':
            return max(0, S - K)
        else:
            return max(0, K - S)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return price


def greeks(S, K, T, r, sigma, option_type='call'):
    """
    Calculate all five Greeks for an option.

    Returns:
        dict with delta, gamma, theta, vega, rho
    """
    if T <= 0:
        return {'delta': 0, 'gamma': 0, 'theta': 0, 'vega': 0, 'rho': 0}

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Delta — sensitivity to stock price
    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1

    # Gamma — rate of change of delta (same for calls and puts)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    # Theta — time decay (per calendar day)
    if option_type == 'call':
        theta = (
            -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        ) / 365
    else:
        theta = (
            -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        ) / 365

    # Vega — sensitivity to volatility (per 1% move in vol)
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100

    # Rho — sensitivity to interest rates (per 1% move in rates)
    if option_type == 'call':
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    return {
        'delta': round(delta, 6),
        'gamma': round(gamma, 6),
        'theta': round(theta, 6),
        'vega':  round(vega, 6),
        'rho':   round(rho, 6)
    }


# ── Quick test ────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

    call_price = black_scholes(S, K, T, r, sigma, 'call')
    put_price  = black_scholes(S, K, T, r, sigma, 'put')
    call_greeks = greeks(S, K, T, r, sigma, 'call')

    print("=" * 50)
    print("BLACK-SCHOLES PRICING ENGINE")
    print("=" * 50)
    print(f"Stock: ${S}  |  Strike: ${K}  |  T: {T}yr  |  r: {r*100}%  |  σ: {sigma*100}%")
    print(f"\nCall Price : ${call_price:.4f}")
    print(f"Put Price  : ${put_price:.4f}")
    print(f"\nGreeks (Call):")
    for name, val in call_greeks.items():
        print(f"  {name.capitalize():<8}: {val}")

    # Put-Call Parity check
    parity = call_price - put_price - S + K * np.exp(-r * T)
    print(f"\nPut-Call Parity check (should be ~0): {parity:.8f}")
    print("=" * 50)

BLACK-SCHOLES PRICING ENGINE
Stock: $100  |  Strike: $100  |  T: 1.0yr  |  r: 5.0%  |  σ: 20.0%

Call Price : $10.4506
Put Price  : $5.5735

Greeks (Call):
  Delta   : 0.636831
  Gamma   : 0.018762
  Theta   : -0.017573
  Vega    : 0.37524
  Rho     : 0.532325

Put-Call Parity check (should be ~0): 0.00000000


In [3]:
import numpy as np
import plotly.graph_objects as go

def monte_carlo_price(S, K, T, r, sigma, option_type='call',
                      n_simulations=10000, n_steps=252, seed=42):
    """
    Price an option via Monte Carlo simulation.

    Simulates n_simulations possible stock price paths,
    calculates payoff at expiration for each, and discounts
    back to present value.

    Parameters:
        S            : Current stock price
        K            : Strike price
        T            : Time to expiration (years)
        r            : Risk-free rate (decimal)
        sigma        : Volatility (decimal)
        option_type  : 'call' or 'put'
        n_simulations: Number of random paths to simulate
        n_steps      : Number of time steps per path
        seed         : Random seed for reproducibility

    Returns:
        price        : Estimated option price
        std_error    : Standard error of the estimate
        paths        : Array of simulated paths (for plotting)
    """
    np.random.seed(seed)

    dt = T / n_steps

    # Simulate all paths at once using vectorized operations
    # Each row is one path, each column is one time step
    random_shocks = np.random.normal(0, 1, (n_simulations, n_steps))

    # Geometric Brownian Motion formula
    daily_returns = np.exp(
        (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * random_shocks
    )

    # Build price paths: start at S, multiply cumulative returns
    price_paths = S * np.cumprod(daily_returns, axis=1)

    # Final prices at expiration
    final_prices = price_paths[:, -1]

    # Calculate payoff for each path
    if option_type == 'call':
        payoffs = np.maximum(final_prices - K, 0)
    else:
        payoffs = np.maximum(K - final_prices, 0)

    # Discount payoffs back to present value
    price     = np.exp(-r * T) * np.mean(payoffs)
    std_error = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_simulations)

    return price, std_error, price_paths


def convergence_analysis(S, K, T, r, sigma, option_type='call',
                         max_simulations=50000):
    """
    Show how Monte Carlo price converges as simulations increase.
    Runs at 10 checkpoints from 100 to max_simulations.
    """
    checkpoints = np.logspace(2, np.log10(max_simulations), 10).astype(int)
    prices = []
    errors = []

    for n in checkpoints:
        price, std_err, _ = monte_carlo_price(
            S, K, T, r, sigma, option_type, n_simulations=n
        )
        prices.append(price)
        errors.append(std_err)

    return checkpoints, prices, errors


def plot_paths(price_paths, S, K, n_display=200):
    """Plot a sample of simulated price paths."""
    fig = go.Figure()

    steps = price_paths.shape[1]
    x = np.linspace(0, 1, steps)

    # Plot sample paths
    for i in range(min(n_display, len(price_paths))):
        fig.add_trace(go.Scatter(
            x=x, y=price_paths[i],
            mode='lines',
            line=dict(width=0.5, color='rgba(96, 165, 250, 0.15)'),
            showlegend=False
        ))

    # Strike price line
    fig.add_hline(y=K, line_dash="dash",
                  line_color="#f87171", line_width=2,
                  annotation_text=f"Strike ${K}")

    # Starting price line
    fig.add_hline(y=S, line_dash="dot",
                  line_color="#34d399", line_width=2,
                  annotation_text=f"Current ${S}")

    fig.update_layout(
        title="Simulated Stock Price Paths",
        xaxis_title="Time (fraction of year)",
        yaxis_title="Stock Price ($)",
        template="plotly_dark",
        height=500,
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117"
    )
    return fig


def plot_convergence(checkpoints, prices, bs_price):
    """Plot Monte Carlo convergence toward Black-Scholes price."""
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=checkpoints, y=prices,
        mode='lines+markers',
        name='Monte Carlo Price',
        line=dict(color='#60a5fa', width=2),
        marker=dict(size=8)
    ))

    fig.add_hline(
        y=bs_price,
        line_dash="dash",
        line_color="#f59e0b",
        line_width=2,
        annotation_text=f"Black-Scholes ${bs_price:.4f}"
    )

    fig.update_layout(
        title="Monte Carlo Convergence to Black-Scholes",
        xaxis_title="Number of Simulations",
        xaxis_type="log",
        yaxis_title="Estimated Option Price ($)",
        template="plotly_dark",
        height=450,
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117"
    )
    return fig


# ── Quick test ────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20

    print("=" * 55)
    print("MONTE CARLO OPTIONS PRICER")
    print("=" * 55)

    for n in [1000, 10000, 50000]:
        price, stderr, _ = monte_carlo_price(
            S, K, T, r, sigma, 'call', n_simulations=n
        )
        print(f"  {n:>6,} simulations → ${price:.4f}  (±{stderr:.4f})")

    print(f"\n  Black-Scholes benchmark  → $10.4506")
    print(f"\n  Convergence: ✓ MC approaches BS as n increases")

    # Convergence plot
    checkpoints, prices, errors = convergence_analysis(
        S, K, T, r, sigma, max_simulations=50000
    )
    fig = plot_convergence(checkpoints, prices, bs_price=10.4506)
    fig.show()

    # Paths plot
    _, _, paths = monte_carlo_price(
        S, K, T, r, sigma, n_simulations=500
    )
    fig2 = plot_paths(paths, S, K)
    fig2.show()

    print("=" * 55)

MONTE CARLO OPTIONS PRICER
   1,000 simulations → $10.3612  (±0.4484)
  10,000 simulations → $10.2211  (±0.1446)
  50,000 simulations → $10.3485  (±0.0654)

  Black-Scholes benchmark  → $10.4506

  Convergence: ✓ MC approaches BS as n increases


In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from scipy.stats import norm
from scipy.optimize import brentq


def get_options_chain(ticker: str):
    """
    Pull live options chain data from Yahoo Finance.
    Returns calls and puts for all available expiration dates.
    """
    stock = yf.Ticker(ticker)

    try:
        spot_price = stock.history(period="1d")['Close'].iloc[-1]
    except Exception:
        raise ValueError(f"Could not fetch price for {ticker}.")

    expirations = stock.options
    if not expirations:
        raise ValueError(f"No options data available for {ticker}.")

    all_calls = []

    for exp_date in expirations[:8]:  # limit to 8 nearest expirations
        try:
            chain = stock.option_chain(exp_date)
            calls = chain.calls.copy()
            calls['expiration'] = exp_date
            all_calls.append(calls)
        except Exception:
            continue

    if not all_calls:
        raise ValueError(f"Could not retrieve options chain for {ticker}.")

    df = pd.concat(all_calls, ignore_index=True)
    return df, spot_price


def implied_volatility(market_price, S, K, T, r, option_type='call'):
    """
    Calculate implied volatility by inverting Black-Scholes.
    Uses Brent's method to find the sigma that produces
    the observed market price.
    """
    if T <= 0 or market_price <= 0:
        return np.nan

    intrinsic = max(0, S - K) if option_type == 'call' else max(0, K - S)
    if market_price < intrinsic * 0.999:
        return np.nan

    def objective(sigma):
        from scipy.stats import norm
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == 'call':
            return (S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)) - market_price
        else:
            return (K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)) - market_price

    try:
        iv = brentq(objective, 1e-6, 20.0, xtol=1e-6, maxiter=500)
        return iv if 0.001 < iv < 20.0 else np.nan
    except Exception:
        return np.nan


def build_vol_surface(ticker='AAPL', r=0.05):
    """
    Build implied volatility surface for a given ticker.
    Returns strikes, expirations, and IV grid for 3D plotting.
    """
    print(f"Fetching options chain for {ticker}...")
    df, spot = get_options_chain(ticker)
    print(f"  Spot price: ${spot:.2f}")
    print(f"  Options contracts loaded: {len(df)}")

    from datetime import datetime
    today = datetime.today()

    records = []
    for _, row in df.iterrows():
        try:
            exp   = datetime.strptime(row['expiration'], '%Y-%m-%d')
            T     = max((exp - today).days / 365.0, 1/365)
            K     = float(row['strike'])
            mid   = (float(row['bid']) + float(row['ask'])) / 2

            if mid <= 0 or row['bid'] == 0:
                continue

            # Only use near-the-money options (80%-120% of spot)
            moneyness = K / spot
            if not (0.80 <= moneyness <= 1.20):
                continue

            iv = implied_volatility(mid, spot, K, T, r, 'call')
            if iv and not np.isnan(iv) and 0.01 < iv < 5.0:
                records.append({
                    'strike':     K,
                    'expiration': row['expiration'],
                    'T':          round(T, 4),
                    'iv':         round(iv, 4),
                    'moneyness':  round(moneyness, 4)
                })
        except Exception:
            continue

    if len(records) < 10:
        raise ValueError(
            f"Not enough valid options data for {ticker}. "
            f"Only {len(records)} valid IVs computed."
        )

    surface_df = pd.DataFrame(records)
    print(f"  Valid IV points computed: {len(surface_df)}")
    return surface_df, spot


def plot_vol_surface(surface_df, spot, ticker):
    """
    Render the implied volatility surface as an interactive 3D plot.
    """
    fig = go.Figure(data=[go.Scatter3d(
        x=surface_df['T'],
        y=surface_df['strike'],
        z=surface_df['iv'] * 100,
        mode='markers',
        marker=dict(
            size=4,
            color=surface_df['iv'] * 100,
            colorscale='Viridis',
            colorbar=dict(title='IV (%)'),
            opacity=0.85
        ),
        hovertemplate=(
            'Expiry: %{x:.2f}yr<br>'
            'Strike: $%{y:.0f}<br>'
            'IV: %{z:.1f}%<br>'
            '<extra></extra>'
        )
    )])

    fig.update_layout(
        title=dict(
            text=f'{ticker} Implied Volatility Surface  |  Spot: ${spot:.2f}',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis_title='Time to Expiry (years)',
            yaxis_title='Strike Price ($)',
            zaxis_title='Implied Volatility (%)',
            bgcolor='#0e1117',
            xaxis=dict(backgroundcolor='#0e1117',
                       gridcolor='#374151', color='white'),
            yaxis=dict(backgroundcolor='#0e1117',
                       gridcolor='#374151', color='white'),
            zaxis=dict(backgroundcolor='#0e1117',
                       gridcolor='#374151', color='white'),
        ),
        paper_bgcolor='#0e1117',
        font=dict(color='white'),
        height=650,
        margin=dict(l=0, r=0, t=60, b=0)
    )
    return fig


# ── Quick test ────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    ticker = 'AAPL'

    print("=" * 55)
    print("IMPLIED VOLATILITY SURFACE")
    print("=" * 55)

    surface_df, spot = build_vol_surface(ticker)

    print(f"\nSample IV data:")
    print(surface_df.sort_values('T').head(10).to_string(index=False))

    fig = plot_vol_surface(surface_df, spot, ticker)
    fig.show()

    print(f"\nVolatility surface rendered for {ticker}")
    print("=" * 55)

IMPLIED VOLATILITY SURFACE
Fetching options chain for AAPL...
  Spot price: $310.85
  Options contracts loaded: 366
  Valid IV points computed: 134

Sample IV data:
 strike expiration      T     iv  moneyness
  285.0 2026-05-27 0.0027 0.7099     0.9168
  302.5 2026-05-27 0.0027 0.2590     0.9731
  305.0 2026-05-27 0.0027 0.2272     0.9812
  255.0 2026-05-29 0.0027 2.0572     0.8203
  270.0 2026-05-29 0.0027 1.5285     0.8686
  280.0 2026-05-29 0.0027 1.2988     0.9008
  282.5 2026-05-29 0.0027 0.8285     0.9088
  290.0 2026-05-29 0.0027 0.6269     0.9329
  292.5 2026-05-29 0.0027 0.5185     0.9410
  295.0 2026-05-29 0.0027 0.6287     0.9490



Volatility surface rendered for AAPL


In [6]:
# ── Cell: strategies.py ───────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def bs_price_simple(S, K, T, r, sigma, opt='call'):
    """Black-Scholes price. Returns 0 for expired options."""
    if T <= 0:
        return max(0, S - K) if opt == 'call' else max(0, K - S)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if opt == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_delta(S, K, T, r, sigma, opt='call'):
    if T <= 0: return 0
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.cdf(d1) if opt == 'call' else norm.cdf(d1) - 1

def bs_gamma(S, K, T, r, sigma):
    if T <= 0: return 0
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def bs_vega(S, K, T, r, sigma):
    if T <= 0: return 0
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T) / 100

# ── Strategy definitions ───────────────────────────────────────────────────────
STRATEGIES = {
    "Long Call": {
        "description": "Buy the right to purchase stock at strike. Unlimited upside, limited downside.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"call","direction":1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'call')}
        ]
    },
    "Long Put": {
        "description": "Buy the right to sell stock at strike. Profits when stock falls.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"put","direction":1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'put')}
        ]
    },
    "Covered Call": {
        "description": "Own the stock, sell a call. Generates income, caps upside.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"stock","direction":1,"K":K,"premium":S},
            {"type":"call","direction":-1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'call')}
        ]
    },
    "Protective Put": {
        "description": "Own the stock, buy a put. Insurance against downside.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"stock","direction":1,"K":K,"premium":S},
            {"type":"put","direction":1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'put')}
        ]
    },
    "Bull Call Spread": {
        "description": "Buy low strike call, sell high strike call. Reduced cost, capped profit.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"call","direction":1, "K":K*0.95,"premium":bs_price_simple(S,K*0.95,T,r,sig,'call')},
            {"type":"call","direction":-1,"K":K*1.05,"premium":bs_price_simple(S,K*1.05,T,r,sig,'call')}
        ]
    },
    "Bear Put Spread": {
        "description": "Buy high strike put, sell low strike put. Profits on moderate decline.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"put","direction":1, "K":K*1.05,"premium":bs_price_simple(S,K*1.05,T,r,sig,'put')},
            {"type":"put","direction":-1,"K":K*0.95,"premium":bs_price_simple(S,K*0.95,T,r,sig,'put')}
        ]
    },
    "Long Straddle": {
        "description": "Buy call and put at same strike. Profits from large moves in either direction.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"call","direction":1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'call')},
            {"type":"put", "direction":1,"K":K,"premium":bs_price_simple(S,K,T,r,sig,'put')}
        ]
    },
    "Iron Condor": {
        "description": "Sell strangle, buy wider strangle. Profits when stock stays in a range.",
        "legs": lambda S, K, T, r, sig: [
            {"type":"put", "direction":1, "K":K*0.85,"premium":bs_price_simple(S,K*0.85,T,r,sig,'put')},
            {"type":"put", "direction":-1,"K":K*0.95,"premium":bs_price_simple(S,K*0.95,T,r,sig,'put')},
            {"type":"call","direction":-1,"K":K*1.05,"premium":bs_price_simple(S,K*1.05,T,r,sig,'call')},
            {"type":"call","direction":1, "K":K*1.15,"premium":bs_price_simple(S,K*1.15,T,r,sig,'call')}
        ]
    }
}

def payoff_at_expiry(legs, stock_prices):
    """
    Calculate total P&L of a multi-leg strategy at expiration
    across a range of stock prices.
    """
    total_pnl = np.zeros(len(stock_prices))
    total_premium = 0

    for leg in legs:
        direction = leg["direction"]
        K_leg     = leg["K"]
        premium   = leg["premium"]
        opt_type  = leg["type"]

        # Net premium paid/received
        total_premium += direction * premium

        for i, S_exp in enumerate(stock_prices):
            if opt_type == 'call':
                intrinsic = max(0, S_exp - K_leg)
            elif opt_type == 'put':
                intrinsic = max(0, K_leg - S_exp)
            else:  # stock
                intrinsic = S_exp - K_leg  # P&L vs purchase price

            total_pnl[i] += direction * intrinsic

    # Subtract net premium paid
    total_pnl -= total_premium
    return total_pnl, total_premium

def strategy_metrics(pnl, stock_prices, S):
    """Calculate max profit, max loss, breakeven points."""
    max_profit = np.max(pnl)
    max_loss   = np.min(pnl)

    # Breakeven: where pnl crosses zero
    breakevens = []
    for i in range(len(pnl) - 1):
        if pnl[i] * pnl[i+1] < 0:  # sign change
            # Linear interpolation
            be = stock_prices[i] - pnl[i] * (stock_prices[i+1] - stock_prices[i]) / (pnl[i+1] - pnl[i])
            breakevens.append(round(be, 2))

    return {
        "max_profit": max_profit if max_profit < 1e6 else float('inf'),
        "max_loss":   max_loss   if max_loss   > -1e6 else float('-inf'),
        "breakevens": breakevens
    }

def plot_strategy(pnl, stock_prices, S, K, strategy_name, legs):
    """Create P&L diagram for the strategy."""
    fig = go.Figure()

    # Zero line
    fig.add_hline(y=0, line_color="#6b7280", line_width=1.5)

    # Current stock price
    fig.add_vline(x=S, line_dash="dash", line_color="#34d399",
                  line_width=1.5, annotation_text=f"Current ${S:.0f}",
                  annotation_position="top left")

    # Strike lines for each leg
    plotted_strikes = set()
    for leg in legs:
        if leg["type"] in ["call","put"] and leg["K"] not in plotted_strikes:
            fig.add_vline(x=leg["K"], line_dash="dot",
                         line_color="#f59e0b", line_width=1,
                         annotation_text=f"K=${leg['K']:.0f}")
            plotted_strikes.add(leg["K"])

    # P&L curve — color by profit/loss
    profit_mask = pnl >= 0
    loss_mask   = pnl < 0

    fig.add_trace(go.Scatter(
        x=stock_prices[profit_mask], y=pnl[profit_mask],
        mode='lines', name='Profit',
        line=dict(color='#34d399', width=3),
        fill='tozeroy', fillcolor='rgba(52, 211, 153, 0.15)'
    ))
    fig.add_trace(go.Scatter(
        x=stock_prices[loss_mask], y=pnl[loss_mask],
        mode='lines', name='Loss',
        line=dict(color='#f87171', width=3),
        fill='tozeroy', fillcolor='rgba(248, 113, 113, 0.15)'
    ))

    fig.update_layout(
        title=f"{strategy_name} — P&L at Expiration",
        xaxis_title="Stock Price at Expiration ($)",
        yaxis_title="Profit / Loss ($)",
        template="plotly_dark",
        height=460,
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117",
        legend=dict(orientation="h", y=1.1),
        margin=dict(l=40, r=20, t=60, b=40)
    )
    return fig

# ── Test ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    S, K, T, r, sig = 100, 100, 0.25, 0.05, 0.20
    stock_range = np.linspace(60, 140, 500)

    print("=" * 55)
    print("OPTIONS STRATEGY BUILDER — Test")
    print("=" * 55)

    for name, strat in STRATEGIES.items():
        legs = strat["legs"](S, K, T, r, sig)
        pnl, net_prem = payoff_at_expiry(legs, stock_range)
        metrics = strategy_metrics(pnl, stock_range, S)
        print(f"\n{name}")
        print(f"  Net premium : ${net_prem:+.4f}")
        max_p_str = 'Unlimited' if metrics['max_profit'] > 999 else f"${metrics['max_profit']:+.2f}"
        max_l_str = 'Unlimited' if metrics['max_loss'] < -999 else f"${metrics['max_loss']:+.2f}"
        print(f"  Max profit  : {max_p_str}")
        print(f"  Max loss    : {max_l_str}")
        print(f"  Breakevens  : {metrics['breakevens']}")

    print("\n" + "=" * 55)

OPTIONS STRATEGY BUILDER — Test

Long Call
  Net premium : $+4.6150
  Max profit  : $+35.39
  Max loss    : $-4.61
  Breakevens  : [np.float64(104.61)]

Long Put
  Net premium : $+3.3728
  Max profit  : $+36.63
  Max loss    : $-3.37
  Breakevens  : [np.float64(96.63)]

Covered Call
  Net premium : $+95.3850
  Max profit  : $-95.39
  Max loss    : $-135.39
  Breakevens  : []

Protective Put
  Net premium : $+103.3728
  Max profit  : $-63.37
  Max loss    : $-103.37
  Breakevens  : []

Bull Call Spread
  Net premium : $+5.2365
  Max profit  : $+4.76
  Max loss    : $-5.24
  Breakevens  : [np.float64(100.24)]

Bear Put Spread
  Net premium : $+4.6393
  Max profit  : $+5.36
  Max loss    : $-4.64
  Breakevens  : [np.float64(100.36)]

Long Straddle
  Net premium : $+7.9878
  Max profit  : $+32.01
  Max loss    : $-7.91
  Breakevens  : [np.float64(92.01), np.float64(107.99)]

Iron Condor
  Net premium : $-3.3505
  Max profit  : $+3.35
  Max loss    : $-6.65
  Breakevens  : [np.float64(91.65

In [7]:
# ── Cell: binomial_tree.py ────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go

def binomial_tree_price(S, K, T, r, sigma, N=100,
                         option_type='call', american=True):
    """
    Cox-Ross-Rubinstein binomial tree option pricer.

    Works for both European and American options.
    American options allow early exercise at every node —
    this is what Black-Scholes fundamentally cannot do.

    Parameters:
        N        : Number of time steps (more = more accurate)
        american : If True, check early exercise at every node
    """
    dt   = T / N
    u    = np.exp(sigma * np.sqrt(dt))         # up factor
    d    = 1 / u                                # down factor
    p    = (np.exp(r * dt) - d) / (u - d)      # risk-neutral probability
    disc = np.exp(-r * dt)                      # discount factor per step

    # Build stock price tree at expiration (terminal nodes only)
    # ST[j] = S * u^j * d^(N-j)
    j = np.arange(N + 1)
    ST = S * (u ** j) * (d ** (N - j))

    # Option payoffs at expiration
    if option_type == 'call':
        values = np.maximum(ST - K, 0)
    else:
        values = np.maximum(K - ST, 0)

    # Backward induction through the tree
    for i in range(N - 1, -1, -1):
        # Roll back one step
        values = disc * (p * values[1:i+2] + (1 - p) * values[0:i+1])

        if american:
            # At each node, stock price is S * u^j * d^(i-j)
            j_nodes = np.arange(i + 1)
            S_nodes = S * (u ** j_nodes) * (d ** (i - j_nodes))

            if option_type == 'call':
                exercise = np.maximum(S_nodes - K, 0)
            else:
                exercise = np.maximum(K - S_nodes, 0)

            # Early exercise if intrinsic > hold value
            values = np.maximum(values, exercise)

    return float(values[0])


def build_tree_for_viz(S, K, T, r, sigma, N=8,
                        option_type='call', american=True):
    """
    Build full stock and option value trees for visualization.
    Limited to small N (<=12) for readability.
    """
    dt   = T / N
    u    = np.exp(sigma * np.sqrt(dt))
    d    = 1 / u
    p    = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    # Stock price tree: stock_tree[i][j] = price at step i, j up-moves
    stock_tree = [[0.0] * (i + 1) for i in range(N + 1)]
    for i in range(N + 1):
        for j in range(i + 1):
            stock_tree[i][j] = S * (u ** j) * (d ** (i - j))

    # Option value tree — start from terminal nodes
    opt_tree = [[0.0] * (i + 1) for i in range(N + 1)]

    # Terminal payoffs
    for j in range(N + 1):
        ST = stock_tree[N][j]
        if option_type == 'call':
            opt_tree[N][j] = max(ST - K, 0)
        else:
            opt_tree[N][j] = max(K - ST, 0)

    # Backward induction
    early_exercise_nodes = []
    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            hold = disc * (p * opt_tree[i+1][j+1] + (1-p) * opt_tree[i+1][j])
            if american:
                S_node = stock_tree[i][j]
                exercise = max(S_node - K, 0) if option_type=='call' \
                           else max(K - S_node, 0)
                if exercise > hold and exercise > 0:
                    opt_tree[i][j] = exercise
                    early_exercise_nodes.append((i, j))
                else:
                    opt_tree[i][j] = hold
            else:
                opt_tree[i][j] = hold

    return stock_tree, opt_tree, early_exercise_nodes, p


def plot_binomial_tree(stock_tree, opt_tree, early_exercise_nodes,
                        option_type, K, N):
    """
    Visualize the binomial tree as a network diagram.
    Nodes colored by option value, early exercise nodes highlighted.
    """
    node_x, node_y = [], []
    node_text, node_color, node_size = [], [], []
    edge_x, edge_y = [], []

    early_set = set(early_exercise_nodes)
    max_opt = max(v for row in opt_tree for v in row if v > 0) or 1

    for i in range(N + 1):
        for j in range(i + 1):
            x = i
            y = j - i / 2  # center vertically
            node_x.append(x)
            node_y.append(y)

            S_val   = stock_tree[i][j]
            opt_val = opt_tree[i][j]

            node_text.append(
                f"Step {i}, {j} up<br>"
                f"S = ${S_val:.2f}<br>"
                f"V = ${opt_val:.3f}"
                + (" ⚡Early exercise" if (i,j) in early_set else "")
            )

            # Color: early exercise = red, else gradient by value
            if (i, j) in early_set:
                node_color.append('#f87171')
            else:
                node_color.append(opt_val / max_opt)

            node_size.append(18 if i < N else 14)

            # Draw edges to children
            if i < N:
                x_child, y_up = i+1, (j+1) - (i+1)/2
                x_child, y_dn = i+1, j     - (i+1)/2
                edge_x += [x, x_child, None, x, x_child, None]
                edge_y += [y, y_up,    None, y, y_dn,    None]

    fig = go.Figure()

    # Edges
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(color='rgba(148,163,184,0.25)', width=1),
        hoverinfo='none', showlegend=False
    ))

    # Nodes (non-early-exercise)
    non_early_mask = [(i,j) not in early_set
                      for i in range(N+1) for j in range(i+1)]
    ne_x = [node_x[k] for k in range(len(node_x)) if non_early_mask[k]]
    ne_y = [node_y[k] for k in range(len(node_y)) if non_early_mask[k]]
    ne_c = [node_color[k] for k in range(len(node_color))
            if non_early_mask[k] and isinstance(node_color[k], float)]
    ne_t = [node_text[k] for k in range(len(node_text)) if non_early_mask[k]]

    fig.add_trace(go.Scatter(
        x=ne_x, y=ne_y, mode='markers',
        marker=dict(
            size=16, color=ne_c,
            colorscale='Blues',
            colorbar=dict(title='Option Value (normalized)'),
            line=dict(color='#60a5fa', width=1)
        ),
        text=ne_t, hoverinfo='text', showlegend=False
    ))

    # Early exercise nodes highlighted separately
    ee_x = [node_x[k] for k in range(len(node_x))
            if not non_early_mask[k]]
    ee_y = [node_y[k] for k in range(len(node_y))
            if not non_early_mask[k]]
    ee_t = [node_text[k] for k in range(len(node_text))
            if not non_early_mask[k]]

    if ee_x:
        fig.add_trace(go.Scatter(
            x=ee_x, y=ee_y, mode='markers',
            marker=dict(size=18, color='#f87171',
                       symbol='star',
                       line=dict(color='white', width=1.5)),
            text=ee_t, hoverinfo='text',
            name='Early Exercise ⚡',
            showlegend=True
        ))

    fig.update_layout(
        title=f"Binomial Tree ({N} steps) — Hover nodes for prices",
        template="plotly_dark",
        height=520,
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                   title="Time Steps →"),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        font=dict(color='white'),
        margin=dict(l=20, r=20, t=60, b=20)
    )
    return fig


# ── Test ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.30

    print("=" * 60)
    print("BINOMIAL TREE OPTION PRICER — CRR Model")
    print("=" * 60)

    for opt in ['call', 'put']:
        bs_val = None
        from scipy.stats import norm
        d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        if opt == 'call':
            bs_val = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
        else:
            bs_val = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

        eu_price = binomial_tree_price(S, K, T, r, sigma, N=200,
                                        option_type=opt, american=False)
        am_price = binomial_tree_price(S, K, T, r, sigma, N=200,
                                        option_type=opt, american=True)
        premium  = am_price - eu_price

        print(f"\n  {opt.upper()} option:")
        print(f"    Black-Scholes (European) : ${bs_val:.4f}")
        print(f"    Binomial     (European)  : ${eu_price:.4f}")
        print(f"    Binomial     (American)  : ${am_price:.4f}")
        print(f"    Early exercise premium   : ${premium:.4f}")

    print("\n  Tree visualization (N=8):")
    st_tree, ov_tree, ee_nodes, p = build_tree_for_viz(
        S, K, T, r, sigma, N=8, option_type='put', american=True
    )
    print(f"    Risk-neutral probability p = {p:.4f}")
    print(f"    Early exercise nodes: {ee_nodes}")
    fig = plot_binomial_tree(st_tree, ov_tree, ee_nodes, 'put', K, N=8)
    fig.show()
    print("=" * 60)

BINOMIAL TREE OPTION PRICER — CRR Model

  CALL option:
    Black-Scholes (European) : $14.2313
    Binomial     (European)  : $14.2165
    Binomial     (American)  : $14.2165
    Early exercise premium   : $0.0000

  PUT option:
    Black-Scholes (European) : $9.3542
    Binomial     (European)  : $9.3395
    Binomial     (American)  : $9.8632
    Early exercise premium   : $0.5237

  Tree visualization (N=8):
    Risk-neutral probability p = 0.5030
    Early exercise nodes: [(7, 0), (7, 1), (7, 2), (7, 3), (6, 0), (6, 1), (6, 2), (5, 0), (5, 1), (4, 0), (3, 0)]


In [8]:
# ── Cell: greeks_heatmap.py ───────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def compute_greeks_grid(S_range, T_range, K, r, sigma, greek='delta', opt='call'):
    """
    Compute a Greek value across a 2D grid of stock prices × time to expiry.
    Returns a matrix ready for heatmap plotting.
    """
    grid = np.zeros((len(T_range), len(S_range)))

    for i, T in enumerate(T_range):
        for j, S in enumerate(S_range):
            if T <= 0:
                grid[i][j] = 0
                continue

            d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
            d2 = d1 - sigma*np.sqrt(T)

            if greek == 'delta':
                grid[i][j] = norm.cdf(d1) if opt=='call' else norm.cdf(d1)-1
            elif greek == 'gamma':
                grid[i][j] = norm.pdf(d1) / (S * sigma * np.sqrt(T))
            elif greek == 'theta':
                term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
                if opt == 'call':
                    grid[i][j] = (term1 - r*K*np.exp(-r*T)*norm.cdf(d2)) / 365
                else:
                    grid[i][j] = (term1 + r*K*np.exp(-r*T)*norm.cdf(-d2)) / 365
            elif greek == 'vega':
                grid[i][j] = S * norm.pdf(d1) * np.sqrt(T) / 100
            elif greek == 'rho':
                if opt == 'call':
                    grid[i][j] = K*T*np.exp(-r*T)*norm.cdf(d2)/100
                else:
                    grid[i][j] = -K*T*np.exp(-r*T)*norm.cdf(-d2)/100

    return grid


def plot_greeks_heatmap(S_range, T_range, grid, greek, opt, K):
    """
    Render Greeks heatmap: stock price on X, time to expiry on Y,
    Greek value as color intensity.
    """
    colorscales = {
        'delta': 'RdBu',
        'gamma': 'Viridis',
        'theta': 'Reds',
        'vega':  'Plasma',
        'rho':   'Cividis'
    }

    fig = go.Figure(data=go.Heatmap(
        x=S_range,
        y=T_range,
        z=grid,
        colorscale=colorscales.get(greek, 'Viridis'),
        colorbar=dict(title=greek.capitalize()),
        hovertemplate=(
            'Stock: $%{x:.1f}<br>'
            'Time: %{y:.2f}yr<br>'
            f'{greek.capitalize()}: %{{z:.4f}}<br>'
            '<extra></extra>'
        )
    ))

    # Strike line
    fig.add_vline(x=K, line_dash="dash",
                  line_color="white", line_width=2,
                  annotation_text=f"Strike K=${K}",
                  annotation_font_color="white")

    greek_descriptions = {
        'delta': 'Delta — How much the option moves per $1 of stock movement',
        'gamma': 'Gamma — Rate of change of Delta (acceleration)',
        'theta': 'Theta — Daily time decay ($/day)',
        'vega':  'Vega — Sensitivity to 1% change in volatility',
        'rho':   'Rho — Sensitivity to 1% change in interest rates'
    }

    fig.update_layout(
        title=f"{greek.capitalize()} Surface — {opt.upper()} | σ={25}% | r=5%<br>"
              f"<sub>{greek_descriptions.get(greek,'')}</sub>",
        xaxis_title="Stock Price ($)",
        yaxis_title="Time to Expiry (years)",
        template="plotly_dark",
        height=500,
        paper_bgcolor="#0e1117",
        plot_bgcolor="#0e1117",
        font=dict(color='white'),
        margin=dict(l=40, r=20, t=80, b=40)
    )
    return fig


# ── Test ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__" or True:
    K, r, sigma = 100, 0.05, 0.25
    S_range = np.linspace(60, 140, 80)
    T_range = np.linspace(0.02, 2.0, 60)

    print("=" * 55)
    print("GREEKS HEATMAP GENERATOR")
    print("=" * 55)

    for greek in ['delta', 'gamma', 'theta', 'vega']:
        grid = compute_greeks_grid(S_range, T_range, K, r, sigma,
                                    greek=greek, opt='call')
        fig = plot_greeks_heatmap(S_range, T_range, grid, greek, 'call', K)
        fig.show()
        print(f"  {greek.capitalize()} heatmap rendered ✓")

    print("=" * 55)

GREEKS HEATMAP GENERATOR


  Delta heatmap rendered ✓


  Gamma heatmap rendered ✓


  Theta heatmap rendered ✓


  Vega heatmap rendered ✓


In [9]:
%%writefile app.py
import streamlit as st
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq
import pandas as pd
import yfinance as yf
from datetime import datetime

st.set_page_config(
    page_title="Options Pricing Engine",
    layout="wide", page_icon="📈",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
  .stApp{background:#0e1117;color:#fafafa}
  [data-testid="stSidebar"]{background:#161b22}
  .metric-card{background:linear-gradient(135deg,#1f2937,#111827);
    border:1px solid #374151;border-radius:12px;
    padding:1.2rem;text-align:center;margin-bottom:.5rem}
  .metric-value{font-size:1.8rem;font-weight:800;color:#60a5fa}
  .metric-label{font-size:.8rem;color:#9ca3af;margin-top:.3rem}
  .greek-card{background:#1f2937;border:1px solid #374151;
    border-radius:8px;padding:.8rem;text-align:center}
  .greek-val{font-size:1.4rem;font-weight:700;color:#34d399}
  .greek-name{font-size:.75rem;color:#9ca3af}
  .sh{background:linear-gradient(90deg,#1e3a5f,#0e1117);
    border-left:4px solid #3b82f6;padding:.6rem 1rem;
    border-radius:0 8px 8px 0;margin:1rem 0 .5rem 0;
    font-size:1rem;font-weight:700;color:#93c5fd}
  .strategy-card{background:#1f2937;border:1px solid #374151;
    border-radius:10px;padding:1rem;margin-bottom:.5rem}
</style>
""", unsafe_allow_html=True)

# ── Core functions ─────────────────────────────────────────────────────────────
def bs_price(S, K, T, r, sigma, opt='call'):
    if T<=0: return max(0,S-K) if opt=='call' else max(0,K-S)
    d1=(np.log(S/K)+(r+.5*sigma**2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T)
    if opt=='call': return S*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
    return K*np.exp(-r*T)*norm.cdf(-d2)-S*norm.cdf(-d1)

def bs_greeks(S,K,T,r,sigma,opt='call'):
    if T<=0: return dict(delta=0,gamma=0,theta=0,vega=0,rho=0)
    d1=(np.log(S/K)+(r+.5*sigma**2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T)
    delta=norm.cdf(d1) if opt=='call' else norm.cdf(d1)-1
    gamma=norm.pdf(d1)/(S*sigma*np.sqrt(T))
    theta=(-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T))
           -r*K*np.exp(-r*T)*(norm.cdf(d2) if opt=='call' else -norm.cdf(-d2)))/365
    vega=S*norm.pdf(d1)*np.sqrt(T)/100
    rho=K*T*np.exp(-r*T)*(norm.cdf(d2) if opt=='call' else -norm.cdf(-d2))/100
    return dict(delta=round(delta,4),gamma=round(gamma,4),
                theta=round(theta,4),vega=round(vega,4),rho=round(rho,4))

def mc_price(S,K,T,r,sigma,opt='call',n=10000):
    np.random.seed(42); dt=T/252
    shocks=np.random.normal(0,1,(n,252))
    paths=S*np.cumprod(np.exp((r-.5*sigma**2)*dt+sigma*np.sqrt(dt)*shocks),axis=1)
    finals=paths[:,-1]
    payoffs=np.maximum(finals-K,0) if opt=='call' else np.maximum(K-finals,0)
    return np.exp(-r*T)*np.mean(payoffs),paths

def implied_vol(mkt,S,K,T,r,opt='call'):
    if T<=0 or mkt<=0: return np.nan
    try: return brentq(lambda sig: bs_price(S,K,T,r,sig,opt)-mkt,1e-6,20.,xtol=1e-6)
    except: return np.nan

def binomial_price(S,K,T,r,sigma,N=150,opt='call',american=True):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u
    p=(np.exp(r*dt)-d)/(u-d); disc=np.exp(-r*dt)
    j=np.arange(N+1); ST=S*(u**j)*(d**(N-j))
    values=np.maximum(ST-K,0) if opt=='call' else np.maximum(K-ST,0)
    for i in range(N-1,-1,-1):
        values=disc*(p*values[1:i+2]+(1-p)*values[0:i+1])
        if american:
            j_n=np.arange(i+1); S_n=S*(u**j_n)*(d**(i-j_n))
            ex=np.maximum(S_n-K,0) if opt=='call' else np.maximum(K-S_n,0)
            values=np.maximum(values,ex)
    return float(values[0])

def build_small_tree(S,K,T,r,sigma,N=8,opt='call',american=True):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u
    p=(np.exp(r*dt)-d)/(u-d); disc=np.exp(-r*dt)
    st=[[S*(u**j)*(d**(i-j)) for j in range(i+1)] for i in range(N+1)]
    ov=[[0.]*(i+1) for i in range(N+1)]
    for j in range(N+1):
        ov[N][j]=max(st[N][j]-K,0) if opt=='call' else max(K-st[N][j],0)
    ee=[]
    for i in range(N-1,-1,-1):
        for j in range(i+1):
            hold=disc*(p*ov[i+1][j+1]+(1-p)*ov[i+1][j])
            ex=(max(st[i][j]-K,0) if opt=='call' else max(K-st[i][j],0)) if american else 0
            if american and ex>hold and ex>0: ov[i][j]=ex; ee.append((i,j))
            else: ov[i][j]=hold
    return st,ov,ee,p

STRATEGIES = {
    "Long Call":      {"desc":"Unlimited upside, limited downside.",
        "legs":lambda S,K,T,r,s:[{"t":"call","d":1,"K":K,"p":bs_price(S,K,T,r,s,'call')}]},
    "Long Put":       {"desc":"Profits when stock falls.",
        "legs":lambda S,K,T,r,s:[{"t":"put","d":1,"K":K,"p":bs_price(S,K,T,r,s,'put')}]},
    "Bull Call Spread":{"desc":"Reduced cost, capped profit on moderate rise.",
        "legs":lambda S,K,T,r,s:[{"t":"call","d":1,"K":K*.95,"p":bs_price(S,K*.95,T,r,s,'call')},
                                   {"t":"call","d":-1,"K":K*1.05,"p":bs_price(S,K*1.05,T,r,s,'call')}]},
    "Bear Put Spread": {"desc":"Profits on moderate decline.",
        "legs":lambda S,K,T,r,s:[{"t":"put","d":1,"K":K*1.05,"p":bs_price(S,K*1.05,T,r,s,'put')},
                                   {"t":"put","d":-1,"K":K*.95,"p":bs_price(S,K*.95,T,r,s,'put')}]},
    "Long Straddle":  {"desc":"Profits from large moves either direction.",
        "legs":lambda S,K,T,r,s:[{"t":"call","d":1,"K":K,"p":bs_price(S,K,T,r,s,'call')},
                                   {"t":"put","d":1,"K":K,"p":bs_price(S,K,T,r,s,'put')}]},
    "Iron Condor":    {"desc":"Profits when stock stays in a range.",
        "legs":lambda S,K,T,r,s:[{"t":"put","d":1,"K":K*.85,"p":bs_price(S,K*.85,T,r,s,'put')},
                                   {"t":"put","d":-1,"K":K*.95,"p":bs_price(S,K*.95,T,r,s,'put')},
                                   {"t":"call","d":-1,"K":K*1.05,"p":bs_price(S,K*1.05,T,r,s,'call')},
                                   {"t":"call","d":1,"K":K*1.15,"p":bs_price(S,K*1.15,T,r,s,'call')}]},
}

def get_pnl(legs, prices):
    pnl=np.zeros(len(prices)); net=0
    for leg in legs:
        net+=leg["d"]*leg["p"]
        for i,S_e in enumerate(prices):
            iv=max(0,S_e-leg["K"]) if leg["t"]=="call" else max(0,leg["K"]-S_e)
            pnl[i]+=leg["d"]*iv
    return pnl-net, net

def greeks_grid(S_rng,T_rng,K,r,sigma,greek,opt):
    G=np.zeros((len(T_rng),len(S_rng)))
    for i,T in enumerate(T_rng):
        for j,S in enumerate(S_rng):
            if T<=0: continue
            d1=(np.log(S/K)+(r+.5*sigma**2)*T)/(sigma*np.sqrt(T)); d2=d1-sigma*np.sqrt(T)
            if greek=='delta': G[i][j]=norm.cdf(d1) if opt=='call' else norm.cdf(d1)-1
            elif greek=='gamma': G[i][j]=norm.pdf(d1)/(S*sigma*np.sqrt(T))
            elif greek=='theta': G[i][j]=(-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T))-r*K*np.exp(-r*T)*(norm.cdf(d2) if opt=='call' else -norm.cdf(-d2)))/365
            elif greek=='vega': G[i][j]=S*norm.pdf(d1)*np.sqrt(T)/100
    return G

# ── Sidebar ────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 📈 Options Pricing Engine")
    st.markdown("*BS · Monte Carlo · Binomial · Vol Surface*")
    st.divider()
    S    =st.number_input("Stock Price ($)",    value=100.,min_value=1.,step=1.)
    K    =st.number_input("Strike Price ($)",   value=100.,min_value=1.,step=1.)
    T    =st.number_input("Time to Expiry (yr)",value=1., min_value=.01,step=.1)
    r    =st.number_input("Risk-Free Rate (%)", value=5., min_value=0., step=.1)/100
    sigma=st.number_input("Volatility (%)",     value=20.,min_value=.1, step=.5)/100
    opt  =st.radio("Option Type",["call","put"])
    n_sim=st.select_slider("MC Simulations",[1000,5000,10000,50000],value=10000)
    st.divider()
    st.markdown("**Nubaid Khan** | Options Pricing Engine")

# ── Tabs ───────────────────────────────────────────────────────────────────────
tabs=st.tabs(["⚡ Pricer","🎲 Monte Carlo","🌋 Vol Surface","📊 Strategies","🌳 Binomial Tree"])

# TAB 1 ── Pricer ──────────────────────────────────────────────────────────────
with tabs[0]:
    st.markdown("# Black-Scholes Pricer")
    st.markdown("*Real-time pricing with full Greek sensitivity analysis and heatmaps*")
    st.divider()

    price=bs_price(S,K,T,r,sigma,opt)
    g=bs_greeks(S,K,T,r,sigma,opt)
    cp=bs_price(S,K,T,r,sigma,'call'); pp=bs_price(S,K,T,r,sigma,'put')

    c1,c2,c3=st.columns(3)
    for col,(label,val) in zip([c1,c2,c3],[
        (f"{opt.upper()} Price",f"${price:.4f}"),
        ("Call Price",f"${cp:.4f}"),
        ("Put Price", f"${pp:.4f}")
    ]):
        col.markdown(f"<div class='metric-card'><div class='metric-value'>{val}</div>"
                     f"<div class='metric-label'>{label}</div></div>",unsafe_allow_html=True)

    parity=cp-pp-S+K*np.exp(-r*T)
    st.markdown(f"**Put-Call Parity:** `{parity:.8f}` ✓")
    st.divider()

    st.markdown("<div class='sh'>The Greeks</div>",unsafe_allow_html=True)
    greek_meta={
        'delta':('Δ Delta','$1 stock → option Δ'),
        'gamma':('Γ Gamma','Rate of change of Δ'),
        'theta':('Θ Theta','Daily decay ($/day)'),
        'vega': ('ν Vega', 'IV +1% → Vega change'),
        'rho':  ('ρ Rho',  'Rate +1% → Rho change')
    }
    cols=st.columns(5)
    for col,(k,(name,desc)) in zip(cols,greek_meta.items()):
        col.markdown(f"<div class='greek-card'><div class='greek-val'>{g[k]}</div>"
                     f"<div class='greek-name'>{name}</div>"
                     f"<div style='font-size:.65rem;color:#6b7280'>{desc}</div></div>",
                     unsafe_allow_html=True)

    st.divider()
    c1,c2=st.columns(2)

    with c1:
        st.markdown("<div class='sh'>Price Sensitivity</div>",unsafe_allow_html=True)
        sense=st.selectbox("Vary:",["Stock Price","Volatility","Time to Expiry"])
        if sense=="Stock Price":
            xs=np.linspace(S*.5,S*1.5,200); ys=[bs_price(x,K,T,r,sigma,opt) for x in xs]; xl="Stock Price ($)"
        elif sense=="Volatility":
            xs=np.linspace(.01,1.,200); ys=[bs_price(S,K,T,r,x,opt) for x in xs]; xl="Volatility"
        else:
            xs=np.linspace(.01,3.,200); ys=[bs_price(S,K,x,r,sigma,opt) for x in xs]; xl="Time (yr)"
        fig=go.Figure(go.Scatter(x=xs,y=ys,mode='lines',line=dict(color='#60a5fa',width=2.5)))
        fig.add_vline(x=xs[100],line_dash="dash",line_color="#f59e0b",line_width=1.5)
        fig.update_layout(xaxis_title=xl,yaxis_title="Price ($)",template="plotly_dark",
                          height=350,paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
                          margin=dict(l=40,r=20,t=20,b=40))
        st.plotly_chart(fig,use_container_width=True)

    with c2:
        st.markdown("<div class='sh'>Greeks Heatmap</div>",unsafe_allow_html=True)
        greek_sel=st.selectbox("Greek:",['delta','gamma','theta','vega'])
        cscales={'delta':'RdBu','gamma':'Viridis','theta':'Reds','vega':'Plasma'}
        S_rng=np.linspace(S*.6,S*1.4,60); T_rng=np.linspace(.05,2.,50)
        G=greeks_grid(S_rng,T_rng,K,r,sigma,greek_sel,opt)
        fig2=go.Figure(go.Heatmap(x=S_rng,y=T_rng,z=G,
                                   colorscale=cscales[greek_sel],
                                   colorbar=dict(title=greek_sel.capitalize()),
                                   hovertemplate='S=$%{x:.1f}<br>T=%{y:.2f}yr<br>Value=%{z:.4f}<extra></extra>'))
        fig2.add_vline(x=K,line_dash="dash",line_color="white",line_width=2,
                       annotation_text=f"K=${K}",annotation_font_color="white")
        fig2.update_layout(xaxis_title="Stock Price ($)",yaxis_title="Time to Expiry (yr)",
                           template="plotly_dark",height=350,
                           paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
                           margin=dict(l=40,r=20,t=20,b=40))
        st.plotly_chart(fig2,use_container_width=True)

# TAB 2 ── Monte Carlo ─────────────────────────────────────────────────────────
with tabs[1]:
    st.markdown("# Monte Carlo Simulator")
    st.divider()
    mc,paths=mc_price(S,K,T,r,sigma,opt,n=n_sim)
    bs_val=bs_price(S,K,T,r,sigma,opt)
    c1,c2,c3=st.columns(3)
    c1.metric("Monte Carlo",f"${mc:.4f}")
    c2.metric("Black-Scholes",f"${bs_val:.4f}")
    c3.metric("Error",f"{abs(mc-bs_val)/bs_val*100:.3f}%")
    st.divider()

    c1,c2=st.columns(2)
    with c1:
        st.markdown("<div class='sh'>Simulated Paths</div>",unsafe_allow_html=True)
        fig3=go.Figure()
        x_ax=np.linspace(0,T,252)
        for i in range(min(250,n_sim)):
            fig3.add_trace(go.Scatter(x=x_ax,y=paths[i],mode='lines',showlegend=False,
                                      line=dict(width=.4,color='rgba(96,165,250,.1)')))
        fig3.add_hline(y=K,line_dash="dash",line_color="#f87171",line_width=2,annotation_text=f"K=${K}")
        fig3.add_hline(y=S,line_dash="dot",line_color="#34d399",line_width=2,annotation_text=f"S=${S}")
        fig3.update_layout(xaxis_title="Time (yr)",yaxis_title="Stock Price ($)",
                           template="plotly_dark",height=380,
                           paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
                           margin=dict(l=40,r=20,t=20,b=40))
        st.plotly_chart(fig3,use_container_width=True)

    with c2:
        st.markdown("<div class='sh'>Convergence to Black-Scholes</div>",unsafe_allow_html=True)
        chk=np.logspace(2,np.log10(n_sim),12).astype(int)
        conv=[mc_price(S,K,T,r,sigma,opt,n=n)[0] for n in chk]
        fig4=go.Figure()
        fig4.add_trace(go.Scatter(x=chk,y=conv,mode='lines+markers',name='MC',
                                   line=dict(color='#60a5fa',width=2),marker=dict(size=7)))
        fig4.add_hline(y=bs_val,line_dash="dash",line_color="#f59e0b",
                       line_width=2,annotation_text=f"BS ${bs_val:.4f}")
        fig4.update_layout(xaxis_title="Simulations",xaxis_type="log",
                           yaxis_title="Price ($)",template="plotly_dark",height=380,
                           paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
                           margin=dict(l=40,r=20,t=20,b=40))
        st.plotly_chart(fig4,use_container_width=True)

# TAB 3 ── Vol Surface ─────────────────────────────────────────────────────────
with tabs[2]:
    st.markdown("# Implied Volatility Surface")
    st.markdown("*Live market data — what the market believes about future uncertainty*")
    st.divider()
    ticker_in=st.text_input("Ticker",value="AAPL").upper().strip()
    r_surf=st.slider("Risk-Free Rate (%)",0.,10.,5.)/100

    if st.button("Generate Surface",type="primary"):
        with st.spinner(f"Fetching {ticker_in} options..."):
            try:
                stock=yf.Ticker(ticker_in)
                spot=stock.history(period="1d")['Close'].iloc[-1]
                today=datetime.today(); records=[]
                for exp in stock.options[:8]:
                    try:
                        chain=stock.option_chain(exp).calls
                        for _,row in chain.iterrows():
                            try:
                                T_e=max((datetime.strptime(exp,'%Y-%m-%d')-today).days/365,1/365)
                                K_e=float(row['strike'])
                                mid=(float(row['bid'])+float(row['ask']))/2
                                if mid<=0 or not(.8<=K_e/spot<=1.2): continue
                                iv=implied_vol(mid,spot,K_e,T_e,r_surf)
                                if iv and not np.isnan(iv) and .01<iv<5.:
                                    records.append({'T':round(T_e,4),'strike':K_e,'iv':round(iv,4)})
                            except: continue
                    except: continue
                if len(records)<10:
                    st.error(f"Not enough data for {ticker_in}")
                else:
                    df_s=pd.DataFrame(records)
                    st.success(f"{ticker_in} | Spot: ${spot:.2f} | {len(df_s)} IV points")
                    fig5=go.Figure(go.Scatter3d(x=df_s['T'],y=df_s['strike'],z=df_s['iv']*100,
                        mode='markers',marker=dict(size=4,color=df_s['iv']*100,
                        colorscale='Viridis',colorbar=dict(title='IV (%)'),opacity=.85),
                        hovertemplate='Expiry: %{x:.2f}yr<br>Strike: $%{y:.0f}<br>IV: %{z:.1f}%<extra></extra>'))
                    fig5.update_layout(title=f'{ticker_in} IV Surface | Spot ${spot:.2f}',
                        scene=dict(xaxis_title='Expiry (yr)',yaxis_title='Strike ($)',
                                   zaxis_title='IV (%)',bgcolor='#0e1117',
                                   xaxis=dict(backgroundcolor='#0e1117',gridcolor='#374151',color='white'),
                                   yaxis=dict(backgroundcolor='#0e1117',gridcolor='#374151',color='white'),
                                   zaxis=dict(backgroundcolor='#0e1117',gridcolor='#374151',color='white')),
                        paper_bgcolor='#0e1117',font=dict(color='white'),height=620,
                        margin=dict(l=0,r=0,t=60,b=0))
                    st.plotly_chart(fig5,use_container_width=True)
            except Exception as e:
                st.error(f"Error: {e}")

# TAB 4 ── Strategy Builder ────────────────────────────────────────────────────
with tabs[3]:
    st.markdown("# Options Strategy Builder")
    st.markdown("*Multi-leg strategy analysis with P&L diagrams at expiration*")
    st.divider()

    strat_name=st.selectbox("Strategy:",list(STRATEGIES.keys()))
    strat=STRATEGIES[strat_name]
    st.info(strat["desc"])

    legs=strat["legs"](S,K,T,r,sigma)
    stock_range=np.linspace(S*.5,S*1.5,500)
    pnl,net_prem=get_pnl(legs,stock_range)

    max_p=np.max(pnl); max_l=np.min(pnl)
    breakevens=[]
    for i in range(len(pnl)-1):
        if pnl[i]*pnl[i+1]<0:
            be=stock_range[i]-pnl[i]*(stock_range[i+1]-stock_range[i])/(pnl[i+1]-pnl[i])
            breakevens.append(round(be,2))

    c1,c2,c3,c4=st.columns(4)
    c1.metric("Net Premium",f"${net_prem:+.4f}")
    c2.metric("Max Profit","Unlimited" if max_p>999 else f"${max_p:+.2f}")
    c3.metric("Max Loss","Unlimited" if max_l<-999 else f"${max_l:+.2f}")
    c4.metric("Breakeven(s)",str(breakevens) if breakevens else "None")

    # Leg details
    st.markdown("<div class='sh'>Position Legs</div>",unsafe_allow_html=True)
    leg_df=pd.DataFrame([{
        "Direction":"BUY" if l["d"]>0 else "SELL",
        "Type":l["t"].upper(),
        "Strike":f"${l['K']:.2f}",
        "Premium":f"${l['p']:.4f}",
        "Net":f"${l['d']*l['p']:+.4f}"
    } for l in legs])
    st.dataframe(leg_df,use_container_width=True,hide_index=True)

    # P&L chart
    st.markdown("<div class='sh'>P&L at Expiration</div>",unsafe_allow_html=True)
    fig6=go.Figure()
    fig6.add_hline(y=0,line_color="#6b7280",line_width=1.5)
    fig6.add_vline(x=S,line_dash="dash",line_color="#34d399",line_width=1.5,
                   annotation_text=f"Current ${S:.0f}",annotation_position="top left",
                   annotation_font_color="#34d399")
    plotted=set()
    for l in legs:
        if l["t"] in ["call","put"] and l["K"] not in plotted:
            fig6.add_vline(x=l["K"],line_dash="dot",line_color="#f59e0b",
                          line_width=1,annotation_text=f"K=${l['K']:.0f}",
                          annotation_font_color="#f59e0b")
            plotted.add(l["K"])
    pm=pnl>=0; lm=pnl<0
    fig6.add_trace(go.Scatter(x=stock_range[pm],y=pnl[pm],mode='lines',name='Profit',
        line=dict(color='#34d399',width=3),fill='tozeroy',fillcolor='rgba(52,211,153,.15)'))
    fig6.add_trace(go.Scatter(x=stock_range[lm],y=pnl[lm],mode='lines',name='Loss',
        line=dict(color='#f87171',width=3),fill='tozeroy',fillcolor='rgba(248,113,113,.15)'))
    fig6.update_layout(xaxis_title="Stock Price at Expiration ($)",yaxis_title="P&L ($)",
                       template="plotly_dark",height=460,paper_bgcolor="#0e1117",
                       plot_bgcolor="#0e1117",legend=dict(orientation="h",y=1.1),
                       margin=dict(l=40,r=20,t=20,b=40))
    st.plotly_chart(fig6,use_container_width=True)

# TAB 5 ── Binomial Tree ───────────────────────────────────────────────────────
with tabs[4]:
    st.markdown("# Binomial Tree Pricer")
    st.markdown("*American option pricing via CRR model — handles early exercise that Black-Scholes cannot*")
    st.divider()

    am_opt=st.radio("Option Type",["call","put"],key="am_opt",horizontal=True)
    n_steps=st.slider("Tree Steps (pricing accuracy)",50,500,150,step=50)
    viz_steps=st.slider("Visualization Steps (readability)",4,12,8)

    eu=binomial_price(S,K,T,r,sigma,N=n_steps,opt=am_opt,american=False)
    am=binomial_price(S,K,T,r,sigma,N=n_steps,opt=am_opt,american=True)
    bs_v=bs_price(S,K,T,r,sigma,am_opt)
    premium=am-eu

    c1,c2,c3,c4=st.columns(4)
    c1.metric("Black-Scholes (European)",f"${bs_v:.4f}")
    c2.metric("Binomial (European)",     f"${eu:.4f}")
    c3.metric("Binomial (American)",     f"${am:.4f}")
    c4.metric("Early Exercise Premium",  f"${premium:.4f}",
              delta=f"{'Worth exercising early' if premium>0.001 else 'No early exercise benefit'}")

    st.divider()

    st.markdown("<div class='sh'>Decision Tree Visualization</div>",unsafe_allow_html=True)
    st.caption("⭐ Red stars = optimal early exercise nodes | Blue = hold | Hover for exact prices")

    st_tree,ov_tree,ee_nodes,p_rn=build_small_tree(S,K,T,r,sigma,N=viz_steps,
                                                     opt=am_opt,american=True)
    early_set=set(ee_nodes)
    max_ov=max(v for row in ov_tree for v in row if v>0) or 1

    node_x,node_y,node_t,node_c=[],[],[],[]
    edge_x,edge_y=[],[]

    for i in range(viz_steps+1):
        for j in range(i+1):
            x=i; y=j-i/2
            node_x.append(x); node_y.append(y)
            node_t.append(f"Step {i} | {j} up-moves<br>"
                          f"S = ${st_tree[i][j]:.2f}<br>"
                          f"V = ${ov_tree[i][j]:.3f}"
                          +(" ⚡ Early Exercise" if (i,j) in early_set else ""))
            node_c.append('#f87171' if (i,j) in early_set
                          else ov_tree[i][j]/max_ov)
            if i<viz_steps:
                y_u=(j+1)-(i+1)/2; y_d=j-(i+1)/2
                edge_x+=[x,i+1,None,x,i+1,None]
                edge_y+=[y,y_u,None,y,y_d,None]

    fig7=go.Figure()
    fig7.add_trace(go.Scatter(x=edge_x,y=edge_y,mode='lines',
        line=dict(color='rgba(148,163,184,.2)',width=1),hoverinfo='none',showlegend=False))

    # Regular nodes
    reg=[k for k in range(len(node_x)) if not isinstance(node_c[k],str)]
    fig7.add_trace(go.Scatter(x=[node_x[k] for k in reg],y=[node_y[k] for k in reg],
        mode='markers',marker=dict(size=16,color=[node_c[k] for k in reg],
        colorscale='Blues',colorbar=dict(title='Value (norm.)'),
        line=dict(color='#60a5fa',width=1)),
        text=[node_t[k] for k in reg],hoverinfo='text',showlegend=False))

    # Early exercise nodes
    ee=[k for k in range(len(node_x)) if isinstance(node_c[k],str)]
    if ee:
        fig7.add_trace(go.Scatter(x=[node_x[k] for k in ee],y=[node_y[k] for k in ee],
            mode='markers',marker=dict(size=20,color='#f87171',symbol='star',
            line=dict(color='white',width=1.5)),
            text=[node_t[k] for k in ee],hoverinfo='text',name='Early Exercise ⚡'))

    fig7.update_layout(
        title=f"CRR Binomial Tree ({viz_steps} steps) | p = {p_rn:.4f} | {'American' if True else 'European'} {am_opt.upper()}",
        template="plotly_dark",height=540,paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
        xaxis=dict(showgrid=False,zeroline=False,showticklabels=False,title="Time Steps →"),
        yaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
        font=dict(color='white'),margin=dict(l=20,r=20,t=60,b=20))
    st.plotly_chart(fig7,use_container_width=True)

    st.caption(f"Risk-neutral probability p = {p_rn:.4f} | "
               f"Early exercise nodes: {len(ee_nodes)} | "
               f"Steps shown: {viz_steps} (pricing used {n_steps})")

Overwriting app.py


In [3]:
!pip install streamlit
import os, time, subprocess

os.system("pkill -f streamlit 2>/dev/null")
time.sleep(2)

subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=open("/tmp/streamlit_options.log","w"),
    stderr=subprocess.STDOUT
)

print("Starting...")
for i in range(15):
    time.sleep(1)
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:8501", timeout=1)
        print(f"Ready after {i+1}s")
        break
    except:
        print(f"Waiting... {i+1}s")

from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8501)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.9 MB/s eta 0:00:00
Starting...
Waiting... 1s
Waiting... 2s
Ready after 3s
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>